In [24]:
import pandas as pd
import numpy as np


In [25]:
# список уникальных событий в логе

data = pd.read_csv('../data/clean_data.csv',parse_dates=['datetime'])
print(f'''Список отслеживаемых событий: 
{data['EventName'].unique()}''')

Список отслеживаемых событий: 
['Tutorial' 'MainScreenAppear' 'OffersScreenAppear' 'CartScreenAppear'
 'PaymentScreenSuccessful']


In [26]:
# отсортируем по пользователю и времени заранее, чтобы не забыть.

data = data.sort_values(by =['DeviceIDHash','datetime'])
#data.head(3)
data.info()
data.head(3)

<class 'pandas.core.frame.DataFrame'>
Index: 241298 entries, 194435 to 218610
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   EventName     241298 non-null  object        
 1   DeviceIDHash  241298 non-null  int64         
 2   ExpId         241298 non-null  int64         
 3   datetime      241298 non-null  datetime64[ns]
 4   date          241298 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 11.0+ MB


,EventName,DeviceIDHash,ExpId,datetime,date
194435,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06
206368,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:54,2019-08-06
206371,PaymentScreenSuccessful,6909561520679493,247,2019-08-06 18:52:58,2019-08-06


In [5]:
# Количество событий без учета групп и количество уникальных пользовтелей по каждому событию
print('События - пользователи')
event_users = (data.groupby('EventName')
               .agg(cnt_event = ('EventName', 'count'), 
                    cnt_uniq_users = ('DeviceIDHash', 'nunique')
                    )
               .sort_values('cnt_event',ascending=False)
               .reset_index()
)
print(event_users)

# А сколько всего уникальных пользователей
print(f''' 
Уникальных пользователей: {data['DeviceIDHash'].nunique()}''')

# Уникальных > Зашедших на стартовую страницу! Учесть это в расчетах

# Доля пользователей, совершивших определенное событие 
ratio_event_users = event_users[['EventName', 'cnt_uniq_users']]
ratio_event_users['ratio'] = round(ratio_event_users['cnt_uniq_users'] / data['DeviceIDHash'].nunique() * 100, 2)
print(ratio_event_users)


События - пользователи
                 EventName  cnt_event  cnt_uniq_users
0         MainScreenAppear     117431            7419
1       OffersScreenAppear      46350            4593
2         CartScreenAppear      42365            3734
3  PaymentScreenSuccessful      34113            3539
4                 Tutorial       1039             840
 
Уникальных пользователей: 7534
                 EventName  cnt_uniq_users  ratio
0         MainScreenAppear            7419  98.47
1       OffersScreenAppear            4593  60.96
2         CartScreenAppear            3734  49.56
3  PaymentScreenSuccessful            3539  46.97
4                 Tutorial             840  11.15


In [8]:
data['EventName'] == 'MainScreenAppear'

194435     True
206368     True
206371    False
206372    False
206373     True
          ...  
218538     True
218576     True
218578     True
218584     True
218610     True
Name: EventName, Length: 241298, dtype: bool

In [6]:
# Получил что всего пользователей 7534, но главный экран видели всего 7419
# Почему количество уникальных не совпадает с количество уникальных, видивших стартовый экран приложения

# перепроверим 
main_users = set(data[(data['EventName'] == 'MainScreenAppear') | (data['EventName'] == 'Tutorial')]['DeviceIDHash']) # уникальные пользователи, имеющие события MainScreenAppear. Добавил в запрос еще и Tutorial после просмотра результата
all_users = set(data['DeviceIDHash']) # все уникальные пользователи
not_main_users = all_users - main_users
print(f'Пользователи, которые не видели MainScreenAppear: {len(not_main_users)}')

# фильтруем пользователей "без основного экрана"
not_main_events = data[data['DeviceIDHash'].isin(not_main_users)]

# какие события встречаются у данных пользователей
not_main_users_events = not_main_events.groupby('DeviceIDHash').agg(
    event_list=('EventName', list),
    expid=('ExpId', 'first'),
    events_count=('EventName', 'count')  
).reset_index()

print("Пользователи, не дошедшие до MainScreenAppear:")
print(not_main_users_events)


# пользователи без покупок
not_payment = not_main_users_events[~not_main_users_events['event_list'].apply(lambda x: 'PaymentScreenSuccessful' in x)]
print(f'Пришедших извне и не сделавших покупок {len(not_payment)}')
not_payment

# вывод: не все пользователи заходят в приложение через ярлык. Есть другие способы (пуш, ссылка от партнеров, смс)

Пользователи, которые не видели MainScreenAppear: 111
Пользователи, не дошедшие до MainScreenAppear:
            DeviceIDHash                                         event_list  \
0      74158328448226259  [PaymentScreenSuccessful, CartScreenAppear, Of...   
1     111394506613435756  [PaymentScreenSuccessful, CartScreenAppear, Of...   
2     214966247576341063  [OffersScreenAppear, PaymentScreenSuccessful, ...   
3     261817378841141406  [PaymentScreenSuccessful, CartScreenAppear, Of...   
4     332529825412858125  [OffersScreenAppear, CartScreenAppear, OffersS...   
..                   ...                                                ...   
106  8586953157808767383  [OffersScreenAppear, CartScreenAppear, Payment...   
107  8804319115517716344  [PaymentScreenSuccessful, CartScreenAppear, Of...   
108  8821171531680573201  [OffersScreenAppear, PaymentScreenSuccessful, ...   
109  9124766629178994679  [OffersScreenAppear, OffersScreenAppear, Offer...   
110  9217594193087726423  [Pay

,DeviceIDHash,event_list,expid,events_count
12,1223708690315846789,[OffersScreenAppear],246,1
15,1478347681767261393,"[OffersScreenAppear, OffersScreenAppear, Offer...",247,4
19,1900791869709139147,[OffersScreenAppear],248,1
20,1958496982439584534,[OffersScreenAppear],248,1
25,2472435690120708424,[OffersScreenAppear],246,1
33,2974131200515436842,"[OffersScreenAppear, OffersScreenAppear]",248,2
46,4111480625662873403,"[OffersScreenAppear, CartScreenAppear, CartScr...",247,3
71,5982189655096173390,"[OffersScreenAppear, OffersScreenAppear, Offer...",248,3
76,6452297315217439503,"[OffersScreenAppear, OffersScreenAppear]",247,2
77,6504780035871286715,[OffersScreenAppear],247,1


In [7]:
# пришедших извне в каждой группе А/А/В
print(not_main_users_events.groupby('expid')['DeviceIDHash'].nunique())

expid
246    34
247    37
248    44
Name: DeviceIDHash, dtype: int64


In [82]:
# малоактивные пользователи менее 2 активностей
counts = data.groupby('DeviceIDHash')['EventName'].transform('count')
one_active = data[counts <= 2]
#print(one_events)
print(' Уникальных малоактивных пользователей')
print(one_active['DeviceIDHash'].nunique())
print(f' С разбивкой по группам {one_active.groupby('ExpId')['DeviceIDHash'].nunique()}')
one_active.groupby('EventName').agg(users = ('DeviceIDHash', 'nunique'), cnt_event = ('EventName', 'count'))


 Уникальных малоактивных пользователей
282
 С разбивкой по группам ExpId
246    92
247    94
248    96
Name: DeviceIDHash, dtype: int64


,users,cnt_event
EventName,,
CartScreenAppear,1,1
MainScreenAppear,271,401
OffersScreenAppear,24,26
Tutorial,16,17


In [7]:
# пользователи, имеющие 1 тип событий. Есть ли такие, сколько их
counts = data.groupby('DeviceIDHash').agg(cnt_events = ('EventName', 'count'), 
                                          uniq_events = ('EventName','nunique'),
                                          name_event = ('EventName', frozenset)
                                          ).reset_index()

one_event = counts[counts['uniq_events']==1].sort_values('cnt_events')
one_event_svod = one_event.groupby('name_event').agg(cnt_user = ('DeviceIDHash', 'count'),
                                                     cnt_event = ('cnt_events', 'sum')
                                                    ).reset_index()
#one_event['cnt_events'].sum() # 40525
print(one_event_svod) # количество пользователей, которые имели 1 тип событий и общее количество сделанных событий
# ого-го! 
bins = [0, 10, 20, 30, 50, 100, np.inf]
labels = ['0-10', '11-20', '21-30', '31-50', '51-100', '100+']
print(one_event.groupby(pd.cut(one_event['cnt_events'], bins=bins, labels = labels)).agg(cnt_users = ('DeviceIDHash', 'count')).reset_index())
#посчитать количество пользователей, которые делали более нескольких десятков обновлений главного экрана. 
#что это? проблема с логом? тестеры, проблемы с ПО, некорректный итоговый отчет?????

             name_event  cnt_user  cnt_event
0    (MainScreenAppear)      2701      40487
1  (OffersScreenAppear)        12         31
2            (Tutorial)         4          7
  cnt_events  cnt_users
0       0-10       1381
1      11-20        694
2      21-30        330
3      31-50        234
4     51-100         71
5       100+          7


C:\Users\User\AppData\Local\Temp\ipykernel_8376\582989410.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(one_event.groupby(pd.cut(one_event['cnt_events'], bins=bins, labels = labels)).agg(cnt_users = ('DeviceIDHash', 'count')).reset_index())


In [5]:
#data.groupby(['ExpId', 'EventName']).agg(cnt_event = ('EventName', 'count'), )

cnt_event
ExpId EventName                         
246   CartScreenAppear             14711
      MainScreenAppear             37708
      OffersScreenAppear           14773
      PaymentScreenSuccessful      11910
      Tutorial                       323
247   CartScreenAppear             12456
      MainScreenAppear             39123
      OffersScreenAppear           15182
      PaymentScreenSuccessful      10043
      Tutorial                       343
248   CartScreenAppear             15198
      MainScreenAppear             40600
      OffersScreenAppear           16395
      PaymentScreenSuccessful      12160
      Tutorial                       373

In [13]:
# Анализ последовательности событий на примере пользователя    214966247576341063 6922444491712477 2614239180094538519  9222603179720523844
data.query('DeviceIDHash == 214966247576341063')

,EventName,DeviceIDHash,ExpId,datetime,date,session
4856,OffersScreenAppear,214966247576341063,248,2019-08-01 07:02:56,2019-08-01,1
4891,PaymentScreenSuccessful,214966247576341063,248,2019-08-01 07:04:00,2019-08-01,1
4893,CartScreenAppear,214966247576341063,248,2019-08-01 07:04:01,2019-08-01,1
4905,CartScreenAppear,214966247576341063,248,2019-08-01 07:04:18,2019-08-01,1
24257,PaymentScreenSuccessful,214966247576341063,248,2019-08-01 15:44:18,2019-08-01,2
24268,PaymentScreenSuccessful,214966247576341063,248,2019-08-01 15:44:28,2019-08-01,2
24269,CartScreenAppear,214966247576341063,248,2019-08-01 15:44:29,2019-08-01,2
53981,PaymentScreenSuccessful,214966247576341063,248,2019-08-02 12:46:40,2019-08-02,3
53984,CartScreenAppear,214966247576341063,248,2019-08-02 12:46:41,2019-08-02,3
53990,OffersScreenAppear,214966247576341063,248,2019-08-02 12:46:49,2019-08-02,3


In [106]:
# пробуем выделить сессии  - неправильный метод, ибо сравнивает с уже имеющейся колонкой, а в ней все нули. А очень хотелось использовать np.select )) Попробовать позже инач сделать

#data1 = data.copy()
#data1['session'] = 0
#limit = pd.Timedelta(minutes = 30)
#conditions = [
#    data1['DeviceIDHash'] != data1.groupby('DeviceIDHash')['DeviceIDHash'].shift(-1),
#    data1['datetime'] <= data1.groupby('DeviceIDHash')['datetime'].shift(-1) + limit,
#    data1['datetime'] > data1.groupby('DeviceIDHash')['datetime'].shift(-1) + limit
#]
#choices = [
#    1,
#    data1.groupby('DeviceIDHash')['session'].shift(-1),
#    data1.groupby('DeviceIDHash')['DeviceIDHash'].shift(-1) + 1
#]
#
#data1['session'] = np.select(conditions,choices)
#data1

,EventName,DeviceIDHash,ExpId,datetime,date,session
194435,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06,1.0
206368,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:54,2019-08-06,0.0
206371,PaymentScreenSuccessful,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,0.0
206372,CartScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,0.0
206373,MainScreenAppear,6909561520679493,247,2019-08-06 18:52:58,2019-08-06,0.0
...,...,...,...,...,...,...
218538,MainScreenAppear,9222603179720523844,248,2019-08-07 09:13:37,2019-08-07,0.0
218576,MainScreenAppear,9222603179720523844,248,2019-08-07 09:14:53,2019-08-07,0.0
218578,MainScreenAppear,9222603179720523844,248,2019-08-07 09:15:01,2019-08-07,0.0
218584,MainScreenAppear,9222603179720523844,248,2019-08-07 09:15:13,2019-08-07,0.0


In [27]:
limit = pd.Timedelta(minutes=30) # зададим лимит бедействия, будет отсекать старую сессию от новой(если бездействие больше указанно времени, значит сессия закончилась и началась новая)

new_user = data['DeviceIDHash'] != data['DeviceIDHash'].shift(1) # помечаем где начинается новый пользователь, с него будет новый счетчик сессий
time_diff = data.groupby('DeviceIDHash')['datetime'].diff() # смотрим разницу по времени с предыдущим событием
timeout = time_diff > limit # помечаем где разница больше лимита на сессию
new_session = new_user | timeout.fillna(True) # новая сессия = либо новый пользователь либо превышен лимит

# Нумеруем сессии ВНУТРИ каждого пользователя
data['session'] = new_session.groupby(data['DeviceIDHash']).cumsum()

# Общее количество сессий на период наблюдений 
print(data.groupby('DeviceIDHash')['session'].max().sum())


45107


In [28]:
data.to_csv('../data/data_sessions.csv', index=False)

In [9]:
data_sessions = (data
                 .groupby(['ExpId', 'DeviceIDHash', 'session'])
                 .agg(time_session = ('datetime', lambda x: (max(x) - min(x)).total_seconds()), # время сессии
                    cnt_pay_session = ('EventName', lambda x: list(x).count('PaymentScreenSuccessful')), # количество покупок за сессию
                    total_actions = ('EventName', lambda x: len(x)), # всего действий за сессию
                    PaySuccess = ('EventName', lambda x: 'PaymentScreenSuccessful' in list(x)), # факт хотя бы одной покупки
                    viewOffers = ('EventName', lambda x: 'OffersScreenAppear' in list(x)),  # просматривают акции
                    from_outside = ('EventName', lambda x: x.iloc[0] not in ['MainScreenAppear', 'Tutorial']) # факт пришел извне
                     )
                 .reset_index()
)
# Строго говоря, невозможно в данном логе узнать время одной сессии. Потому что лог отображает когда мы совершили действие, 
# то есть мы знаем когда пользователь совершил последнее действие (например, перешел в основное окно после покупки), но у нас нет факта когда клиент вышел из программы совсем.
# Поэтому  будут сессси со временем = 0 секунд (когда в сессии всего 1 действие)
data_sessions

# посмотреть те сессии, где время = 0 (одна активность за сессию). Если такое действие встречается часто и у активных пользователей, 
# то может означать, что часто вылетает программа при входе                                             

,ExpId,DeviceIDHash,session,time_session,cnt_pay_session,total_actions,PaySuccess,viewOffers,from_outside
0,246,6888746892508752,1,0.0,0,1,False,False,False
1,246,6922444491712477,1,13.0,1,5,True,True,False
2,246,6922444491712477,2,16.0,1,5,True,True,False
3,246,6922444491712477,3,335.0,0,6,False,True,False
4,246,6922444491712477,4,925.0,5,17,True,True,False
...,...,...,...,...,...,...,...,...,...
45102,248,9222603179720523844,7,0.0,0,1,False,False,False
45103,248,9222603179720523844,8,1628.0,0,10,False,False,False
45104,248,9222603179720523844,9,840.0,0,6,False,False,False
45105,248,9222603179720523844,10,624.0,0,2,False,False,False


In [19]:
data_sessions.query('time_session == 0') # получилось 7849 сессий где было сделано одно действие, либо несколько событий в одно время
# такое возможно, после продажи, когда автоматом приложение переходит в корзину, и если она пуста, то на главный экран.
data_sessions.query("(time_session == 0) & (PaySuccess == True) & (total_actions == 1)") # 7 сессий с 1 действием - продажа. Как такое возможно? Логи не полные или есть такая возможность, 
# что делается продажа напрямую из внешней ссылки... тут нужно изучать работу ПО или обращаться к тех.поддержке с разъяснениями

,ExpId,DeviceIDHash,session,time_session,cnt_pay_session,total_actions,PaySuccess,viewOffers,from_outside
314,246,183963844301624746,5,0.0,1,1,True,False,True
7788,246,4906928148806627393,11,0.0,1,1,True,False,True
14368,246,9110608674894233944,11,0.0,1,1,True,False,True
17303,247,1743071356190500664,9,0.0,1,1,True,False,True
30029,248,74248254043264762,15,0.0,1,1,True,False,True
31625,248,943476888243120350,5,0.0,1,1,True,False,True
42358,248,7483596365340385091,10,0.0,1,1,True,False,True


In [18]:
# хочу понять, обязательно ли зайти в корзину, чтобы сделать покупку или можно сделать это без захода. Чтобы попробовать выстроить воронку продаж
# то есть нужно найти 
# либо сессии, где есть покупки и факт захода в корзину, НО время корзины должно быть более ранним, чем покупка
# либо сессии, где есть покупки, но нет заходов в корзину

# сгруппируем по сессиям и выпишем все события и их время
sesion_event_time = data.groupby(['DeviceIDHash', 'session']).agg(event_name = ('EventName', list),
                                                                 event_time = ('datetime', list)
                                                                 )


In [22]:
# есть ли сессии когда была продажа, но не было корзины
sesion_event_time[sesion_event_time['event_name'].apply(lambda x: ('PaymentScreenSuccessful' in x) and ('CartScreenAppear' not in x))]

# есть такие продажи. значит заход в корзину для покупки необязателен. Если это не так, то значит проблемы с логированием  

# то есть цепочка "заход в ПО -> Корзина -> Покупка" не является обязательной. Не получается линейная воронка

,,event_name,event_time
DeviceIDHash,session,,
74248254043264762,15,[PaymentScreenSuccessful],[2019-08-07 16:32:20]
77364241990273403,11,"[MainScreenAppear, PaymentScreenSuccessful, Pa...","[2019-08-05 22:38:50, 2019-08-05 22:38:52, 201..."
91292479590032512,6,"[MainScreenAppear, PaymentScreenSuccessful, Of...","[2019-08-02 17:37:26, 2019-08-02 17:37:37, 201..."
170708003193483797,6,"[MainScreenAppear, PaymentScreenSuccessful, Of...","[2019-08-04 06:35:29, 2019-08-04 06:36:09, 201..."
183963844301624746,5,[PaymentScreenSuccessful],[2019-08-03 13:06:22]
...,...,...,...
8494595990824157619,3,"[MainScreenAppear, PaymentScreenSuccessful]","[2019-08-07 05:44:34, 2019-08-07 05:44:40]"
8714004768247930523,1,"[MainScreenAppear, PaymentScreenSuccessful, Of...","[2019-08-01 16:32:07, 2019-08-01 16:32:08, 201..."
8726413942744351490,4,"[MainScreenAppear, PaymentScreenSuccessful, Of...","[2019-08-01 16:15:40, 2019-08-01 16:15:41, 201..."


In [21]:
# для тестов, анализа
data[data['DeviceIDHash'] == 943476888243120350]

,EventName,DeviceIDHash,ExpId,datetime,date,session
87339,Tutorial,943476888243120350,248,2019-08-03 12:10:10,2019-08-03,1
87364,MainScreenAppear,943476888243120350,248,2019-08-03 12:10:38,2019-08-03,1
87369,MainScreenAppear,943476888243120350,248,2019-08-03 12:10:44,2019-08-03,1
87461,MainScreenAppear,943476888243120350,248,2019-08-03 12:13:12,2019-08-03,1
89192,MainScreenAppear,943476888243120350,248,2019-08-03 12:59:00,2019-08-03,2
89197,CartScreenAppear,943476888243120350,248,2019-08-03 12:59:15,2019-08-03,2
89198,MainScreenAppear,943476888243120350,248,2019-08-03 12:59:15,2019-08-03,2
89213,OffersScreenAppear,943476888243120350,248,2019-08-03 12:59:38,2019-08-03,2
89437,PaymentScreenSuccessful,943476888243120350,248,2019-08-03 13:05:04,2019-08-03,2
89439,OffersScreenAppear,943476888243120350,248,2019-08-03 13:05:05,2019-08-03,2


In [142]:
# data.query("EventName == 'Tutorial' and session > 1") # пользуются ли справкой НЕ при первом использовании ПО
#
#data_sessions['from_outside'].sum() # 5507

# разбивка по группам
type_entrance = data_sessions.groupby(['ExpId','from_outside', 'viewOffers']).agg(Entrance = ('from_outside','count'), 
                                                          Uniq_users = ('DeviceIDHash', 'nunique'),
                                                          #viewOffer = ('viewOffers', 'sum'), 
                                                          donePay = ('PaySuccess', 'sum'), 
                                                          totalPay = ('cnt_pay_session', 'sum')
                                                          )#.reset_index()
type_entrance.to_clipboard(index = False, sep=' ')
type_entrance

Entrance  Uniq_users  donePay  totalPay
ExpId from_outside viewOffers                                         
246   False        False           6466        1782       93       141
                   True            6367        1455     3247      8971
      True         False            150         104      121       217
                   True            1614         568      746      2581
247   False        False           7053        1808      100       130
                   True            6354        1429     3086      6871
      True         False            137         112      114       236
                   True            1759         563      777      2806
248   False        False           6795        1829       96       118
                   True            6565        1437     3251      7977
      True         False            186         115      154       282
                   True            1661         606      770      3783

In [144]:
# Дерево событий (попытка описать воронку продаж в разрезе сессий, типа захода в ПО и просмотра акций)
type_entrance = data_sessions.groupby(['from_outside', 'viewOffers']).agg(Entrance = ('from_outside','count'), 
                                                          Uniq_users = ('DeviceIDHash', 'nunique'),
                                                          #viewOffer = ('viewOffers', 'sum'), 
                                                          donePay = ('PaySuccess', 'sum'), 
                                                          totalPay = ('cnt_pay_session', 'sum')
                                                          ).reset_index()
type_entrance.to_clipboard(index = False, sep=' ')
type_entrance

,from_outside,viewOffers,Entrance,Uniq_users,donePay,totalPay
0,False,False,20314,5419,289,389
1,False,True,19286,4321,9584,23819
2,True,False,473,331,389,735
3,True,True,5034,1737,2293,9170


In [ ]:
# нужно сравнить группы А и А - равны ли их основные характеристики, чтобы потом сравнить их с группой В

# пользователь 9222603179720523844 имеет только событие MainScreenAppear и больше ничего!, хотя время проводит много в приложении. Кто он, сколько таких? 
# Ошибка приложения, тестовый пользователь 